# Whale Data Pipeline (1m)

This notebook prepares a rough **1-minute whale dataset** from raw Hyperliquid whale alerts and whale positions so it can be handed to the existing `Data_Pipeline` / `Feature_Engineering` workflow.

Stages:
1. Configure input paths and archetype filter
2. Standardize raw whale parquet files
3. Build wallet archetypes using the existing 5-archetype heuristic
4. Aggregate filtered BTC whale activity to 1-minute features
5. Left-merge the whale table into `cleaned_1m.csv`
6. Save merge-ready outputs


## Expected Outputs

This notebook writes three handoff tables:

- `data/whale_processed/wallet_archetypes.csv`
- `data/whale_processed/whale_features_1m.csv`
- `data/merged/cleaned_1m_with_whales.csv`

Notes:
- The whale base table is intentionally rough and **1-minute only**.
- Archetypes are computed from multi-symbol wallet behavior, but the final minute table is filtered to `TARGET_SYMBOL`.
- The actual market merge is a final handoff step; the main product is the whale-side base table.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd().resolve()
if not (ROOT / "tool").exists() and (ROOT.parent / "tool").exists():
    ROOT = ROOT.parent
print("Project root:", ROOT)


def discover_latest_raw_whale_dir(base_dir: Path) -> Path:
    candidates = sorted([p for p in base_dir.glob("raw_whales_*") if p.is_dir()])
    for path in reversed(candidates):
        if (path / "whale_alerts_raw.parquet").exists() and (path / "whale_positions_raw.parquet").exists():
            return path
    raise FileNotFoundError(f"No raw whale parquet bundle found under {base_dir}")


RAW_WHALE_DIR = discover_latest_raw_whale_dir(ROOT / "data" / "raw_whales")
ALERTS_PATH = RAW_WHALE_DIR / "whale_alerts_raw.parquet"
POSITIONS_PATH = RAW_WHALE_DIR / "whale_positions_raw.parquet"
MARKET_1M_PATH = ROOT / "data" / "cleaned" / "cleaned_1m.csv"

TARGET_SYMBOL = "BTC"
FOCUS_SYMBOLS_FOR_ARCHETYPES = ["BTC", "ETH", "HYPE", "SOL"]
SELECTED_ARCHETYPES = ["all"]

TOP_EXPOSURE_WALLETS = 125
TOP_TURNOVER_WALLETS = 125
MIN_WALLET_SNAPSHOTS = 720
TRADE_EPS_USD = 50_000
POSITION_BATCH_SIZE = 400_000

OUTPUT_DIR = ROOT / "data" / "whale_processed"
MERGED_OUTPUT_DIR = ROOT / "data" / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MERGED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHETYPE_ORDER = [
    "high_conviction",
    "fast_speculator",
    "hedger_like",
    "fragile",
    "pnl_swing",
]

assert ALERTS_PATH.exists(), f"Missing alerts parquet: {ALERTS_PATH}"
assert POSITIONS_PATH.exists(), f"Missing positions parquet: {POSITIONS_PATH}"

config_view = pd.DataFrame(
    {
        "setting": [
            "RAW_WHALE_DIR",
            "ALERTS_PATH",
            "POSITIONS_PATH",
            "MARKET_1M_PATH",
            "TARGET_SYMBOL",
            "FOCUS_SYMBOLS_FOR_ARCHETYPES",
            "SELECTED_ARCHETYPES",
            "TOP_EXPOSURE_WALLETS",
            "TOP_TURNOVER_WALLETS",
            "MIN_WALLET_SNAPSHOTS",
            "TRADE_EPS_USD",
            "POSITION_BATCH_SIZE",
        ],
        "value": [
            str(RAW_WHALE_DIR),
            str(ALERTS_PATH),
            str(POSITIONS_PATH),
            str(MARKET_1M_PATH),
            TARGET_SYMBOL,
            ", ".join(FOCUS_SYMBOLS_FOR_ARCHETYPES),
            ", ".join(SELECTED_ARCHETYPES),
            TOP_EXPOSURE_WALLETS,
            TOP_TURNOVER_WALLETS,
            MIN_WALLET_SNAPSHOTS,
            TRADE_EPS_USD,
            POSITION_BATCH_SIZE,
        ],
    }
)
display(config_view)


## 1) Helper Functions

These helpers are adapted from:

- `Notebook_Explorations/Hyperliquid_Whale_Wallet_Exploration.ipynb`
- `Notebook_Explorations/BTC_30m_Hyperliquid_Whale_Research_Pipeline.ipynb`

The goal here is to keep the notebook self-contained while preserving the same canonicalization and archetype logic.


In [2]:
def pick_col(columns, patterns):
    cols = list(columns)
    for pattern in patterns:
        regex = re.compile(pattern, re.IGNORECASE)
        for col in cols:
            if regex.search(str(col)):
                return col
    return None


def to_datetime_utc(series: pd.Series) -> pd.Series:
    raw = pd.Series(series)
    num = pd.to_numeric(raw, errors="coerce")
    if num.notna().any():
        median_value = num.dropna().median()
        if median_value > 1e12:
            return pd.to_datetime(num, unit="ms", utc=True, errors="coerce")
        if median_value > 1e9:
            return pd.to_datetime(num, unit="s", utc=True, errors="coerce")
    return pd.to_datetime(raw, utc=True, errors="coerce")


def coalesce_numeric(df: pd.DataFrame, cols) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype="float64")
    for col in cols:
        if col and col in df.columns:
            series = pd.to_numeric(df[col], errors="coerce")
            out = out.fillna(series)
    return out


def coalesce_text(df: pd.DataFrame, cols) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype="object")
    for col in cols:
        if col and col in df.columns:
            series = df[col].astype("string")
            mask = out.isna() & series.notna()
            out.loc[mask] = series.loc[mask]
    return out.astype("string")


def coalesce_datetime(df: pd.DataFrame, cols) -> pd.Series:
    out = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")
    for col in cols:
        if col and col in df.columns:
            series = to_datetime_utc(df[col])
            out = out.fillna(series)
    return out


def safe_divide(num, den):
    if isinstance(den, (pd.Series, pd.DataFrame)):
        den_safe = den.replace(0, np.nan)
    else:
        den_safe = np.where(np.asarray(den) == 0, np.nan, den)
    return num / den_safe


def weighted_average(series: pd.Series, weights: pd.Series) -> float:
    mask = series.notna() & weights.notna() & (weights > 0)
    if not mask.any():
        return np.nan
    return float(np.average(series[mask], weights=weights[mask]))


def infer_side_sign(
    df: pd.DataFrame,
    numeric_candidates=None,
    text_candidates=None,
    action_col: str | None = None,
) -> pd.Series:
    out = pd.Series(0, index=df.index, dtype="int64")

    for col in numeric_candidates or []:
        if col and col in df.columns:
            numeric = pd.to_numeric(df[col], errors="coerce")
            sign = np.sign(numeric).fillna(0).astype(int)
            if sign.ne(0).any():
                out = out.where(out != 0, sign)
                break

    if out.eq(0).all():
        for col in text_candidates or []:
            if col and col in df.columns:
                text = df[col].astype(str).str.lower()
                long_mask = text.str.contains("long|buy|bid", na=False)
                short_mask = text.str.contains("short|sell|ask", na=False)
                temp = pd.Series(np.select([long_mask, short_mask], [1, -1], default=0), index=df.index)
                out = out.where(out != 0, temp)
                if out.ne(0).any():
                    break

    if action_col and action_col in df.columns and out.ne(0).any():
        action_text = df[action_col].astype(str).str.lower()
        action_num = pd.to_numeric(df[action_col], errors="coerce")
        is_open = action_text.str.contains("open|increase|add|build|long|short", na=False)
        is_close = action_text.str.contains("close|reduce|decrease|trim", na=False)
        if not is_open.any() and not is_close.any():
            is_open = action_num.eq(1)
            is_close = action_num.eq(2)
        out = pd.Series(np.where(is_close & out.ne(0), -out, out), index=df.index)
        out = pd.Series(np.where(is_open & out.ne(0), out, out), index=df.index)

    return out.astype(int)


def normalize_action_label(series: pd.Series) -> pd.Series:
    text = pd.Series(series).astype("string").str.lower()
    action_num = pd.to_numeric(series, errors="coerce")
    out = pd.Series("unknown", index=text.index, dtype="string")

    open_mask = text.str.contains("open|increase|add|build", na=False) | action_num.eq(1)
    close_mask = text.str.contains("close|reduce|decrease|trim", na=False) | action_num.eq(2)
    long_mask = text.str.contains("long|buy|bid", na=False)
    short_mask = text.str.contains("short|sell|ask", na=False)

    out.loc[open_mask] = "open"
    out.loc[close_mask] = "close"
    out.loc[~open_mask & ~close_mask & long_mask] = "long"
    out.loc[~open_mask & ~close_mask & short_mask] = "short"
    return out.fillna("unknown")


def datetime_to_time_ms(series: pd.Series) -> pd.Series:
    s = pd.to_datetime(series, utc=True, errors="coerce")
    out = pd.Series(pd.NA, index=s.index, dtype="Int64")
    mask = s.notna()
    if mask.any():
        out.loc[mask] = (s.loc[mask].astype("int64") // 1_000_000).astype("int64")
    return out


def iter_parquet_batches(path: Path, columns=None, batch_size: int = POSITION_BATCH_SIZE):
    parquet_file = pq.ParquetFile(path)
    for batch in parquet_file.iter_batches(batch_size=batch_size, columns=columns):
        yield batch.to_pandas()


def update_turnover_from_chunk(chunk: pd.DataFrame, last_signed: dict[tuple[str, str], float]) -> pd.Series:
    if chunk.empty:
        return pd.Series(dtype="float64")
    chunk = chunk.sort_values(["event_time", "user", "symbol"]).reset_index(drop=True).copy()
    chunk["prev_signed_position_usd"] = chunk.groupby(["user", "symbol"])["signed_position_usd"].shift(1)
    first_mask = ~chunk.duplicated(["user", "symbol"])
    first_idx = chunk.index[first_mask]
    if len(first_idx):
        prev_vals = [
            last_signed.get((row.user, row.symbol), np.nan)
            for row in chunk.loc[first_idx, ["user", "symbol"]].itertuples(index=False)
        ]
        chunk.loc[first_idx, "prev_signed_position_usd"] = prev_vals
    chunk["delta_position_usd"] = (chunk["signed_position_usd"] - chunk["prev_signed_position_usd"]).fillna(0.0)
    turnover = (
        chunk.loc[chunk["delta_position_usd"].abs() >= TRADE_EPS_USD]
        .groupby("user")["delta_position_usd"]
        .apply(lambda s: float(s.abs().sum()))
    )
    last_rows = chunk.groupby(["user", "symbol"], sort=False).tail(1)
    for row in last_rows[["user", "symbol", "signed_position_usd"]].itertuples(index=False):
        last_signed[(row.user, row.symbol)] = float(row.signed_position_usd)
    return turnover


def standardize_alerts(df: pd.DataFrame) -> pd.DataFrame:
    alert_time_col = pick_col(df.columns, [r"create_time", r"event_time", r"timestamp", r"time", r"ts", r"created"])
    action_col = pick_col(df.columns, [r"position_action", r"action", r"dir", r"side", r"type"])
    side_col = pick_col(df.columns, [r"side", r"direction", r"positionSide", r"long|short"])
    size_col = pick_col(df.columns, [r"^position_size$", r"size", r"amount", r"qty"])
    value_col = pick_col(df.columns, [r"position_value_usd", r"notional", r"value", r"usd"])
    entry_col = pick_col(df.columns, [r"entry_price", r"price"])
    liq_col = pick_col(df.columns, [r"liq_price", r"liquidation"])
    user_col = pick_col(df.columns, [r"^user$", r"wallet", r"address", r"account", r"trader"])
    symbol_col = pick_col(df.columns, [r"^symbol$", r"coin", r"asset", r"pair", r"token"])

    out = pd.DataFrame(index=df.index)
    out["ingested_at"] = coalesce_datetime(df, ["ingested_at"])
    out["create_time"] = coalesce_datetime(df, [alert_time_col])
    out["event_time"] = coalesce_datetime(df, [alert_time_col, "ingested_at"])
    out["user"] = coalesce_text(df, [user_col]).fillna("unknown_user")
    out["symbol"] = coalesce_text(df, [symbol_col]).str.upper()
    out["position_size"] = coalesce_numeric(df, [size_col])
    out["entry_price"] = coalesce_numeric(df, [entry_col])
    out["liq_price"] = coalesce_numeric(df, [liq_col])
    out["position_value_usd"] = coalesce_numeric(df, [value_col])
    missing_value = out["position_value_usd"].isna() & out["position_size"].notna() & out["entry_price"].notna()
    out.loc[missing_value, "position_value_usd"] = (
        out.loc[missing_value, "position_size"].abs() * out.loc[missing_value, "entry_price"].abs()
    )
    out["position_action"] = coalesce_text(df, [action_col, side_col]).fillna("unknown")
    out["position_action_label"] = normalize_action_label(out["position_action"])
    out["side_sign"] = infer_side_sign(
        df,
        numeric_candidates=[size_col, value_col],
        text_candidates=[side_col, action_col],
        action_col=action_col,
    )
    out["abs_notional_usd"] = out["position_value_usd"].abs()
    out["signed_notional_usd"] = out["abs_notional_usd"].fillna(0.0) * out["side_sign"].fillna(0)
    out = out.dropna(subset=["event_time", "symbol"]).sort_values(["user", "symbol", "event_time"]).reset_index(drop=True)
    return out


def standardize_positions(df: pd.DataFrame) -> pd.DataFrame:
    time_col = pick_col(df.columns, [r"update_time", r"create_time", r"ingested_at", r"timestamp", r"time", r"ts"])
    action_col = pick_col(df.columns, [r"position_action", r"action", r"dir"])
    side_col = pick_col(df.columns, [r"side", r"direction", r"positionSide", r"long|short"])
    size_col = pick_col(df.columns, [r"^position_size$", r"size", r"amount", r"qty"])
    value_col = pick_col(df.columns, [r"position_value_usd", r"notional", r"value", r"usd"])
    entry_col = pick_col(df.columns, [r"entry_price"])
    mark_col = pick_col(df.columns, [r"mark_price", r"mark", r"price"])
    liq_col = pick_col(df.columns, [r"liq_price", r"liquidation"])
    lev_col = pick_col(df.columns, [r"leverage"])
    margin_col = pick_col(df.columns, [r"margin_balance", r"margin"])
    pnl_col = pick_col(df.columns, [r"unrealized_pnl", r"pnl", r"profit"])
    funding_col = pick_col(df.columns, [r"funding_fee", r"funding"])
    mode_col = pick_col(df.columns, [r"margin_mode", r"mode"])
    user_col = pick_col(df.columns, [r"^user$", r"wallet", r"address", r"account", r"trader"])
    symbol_col = pick_col(df.columns, [r"^symbol$", r"coin", r"asset", r"pair", r"token"])

    out = pd.DataFrame(index=df.index)
    out["ingested_at"] = coalesce_datetime(df, ["ingested_at"])
    out["event_time"] = coalesce_datetime(df, [time_col, "ingested_at"])
    out["user"] = coalesce_text(df, [user_col]).fillna("unknown_user")
    out["symbol"] = coalesce_text(df, [symbol_col]).str.upper()
    out["position_size"] = coalesce_numeric(df, [size_col])
    out["entry_price"] = coalesce_numeric(df, [entry_col])
    out["mark_price"] = coalesce_numeric(df, [mark_col])
    out["liq_price"] = coalesce_numeric(df, [liq_col])
    out["leverage"] = coalesce_numeric(df, [lev_col])
    out["margin_balance"] = coalesce_numeric(df, [margin_col])
    out["unrealized_pnl"] = coalesce_numeric(df, [pnl_col])
    out["funding_fee"] = coalesce_numeric(df, [funding_col])
    out["margin_mode"] = coalesce_text(df, [mode_col]).fillna("unknown")
    out["position_value_usd"] = coalesce_numeric(df, [value_col])
    missing_value = out["position_value_usd"].isna() & out["position_size"].notna() & out["mark_price"].notna()
    out.loc[missing_value, "position_value_usd"] = (
        out.loc[missing_value, "position_size"].abs() * out.loc[missing_value, "mark_price"].abs()
    )
    out["side_sign"] = infer_side_sign(
        df,
        numeric_candidates=[size_col],
        text_candidates=[side_col, action_col],
        action_col=action_col,
    )
    out["abs_position_usd"] = out["position_value_usd"].abs()
    out["signed_position_usd"] = out["abs_position_usd"].fillna(0.0) * out["side_sign"].fillna(0)
    out["liq_gap_pct"] = safe_divide(out["liq_price"] - out["mark_price"], out["mark_price"])
    out["liq_distance_pct"] = out["liq_gap_pct"].abs()
    out["pnl_pct"] = safe_divide(out["unrealized_pnl"], out["abs_position_usd"]).replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["event_time", "symbol"]).sort_values(["user", "symbol", "event_time"]).reset_index(drop=True)
    out = out.drop_duplicates(subset=["user", "symbol", "event_time"], keep="last").reset_index(drop=True)
    return out


def compute_wallet_features(focus_positions: pd.DataFrame, wallet_minute: pd.DataFrame) -> pd.DataFrame:
    active_wallet_minute = wallet_minute.loc[wallet_minute["total_abs_exposure"] > 0].copy()
    wallet_trade_flags = (
        focus_positions.groupby(["minute", "user"])["is_trade_minute"]
        .max()
        .rename("trade_flag")
        .reset_index()
    )

    turnover = (
        focus_positions.loc[focus_positions["is_trade_minute"]]
        .groupby("user")["abs_delta_position_usd"]
        .sum()
        .rename("turnover_usd")
    )
    avg_abs_exposure = active_wallet_minute.groupby("user")["total_abs_exposure"].mean().rename("avg_abs_exposure")
    avg_leverage = (
        active_wallet_minute.groupby("user")
        .apply(lambda g: weighted_average(g["avg_leverage"], g["total_abs_exposure"]))
        .rename("avg_leverage")
    )
    wallet_pnl_pct = safe_divide(active_wallet_minute["total_unrealized_pnl"], active_wallet_minute["total_abs_exposure"])
    pnl_volatility = wallet_pnl_pct.groupby(active_wallet_minute["user"]).std().rename("pnl_volatility")
    median_liq_distance_pct = (
        focus_positions.loc[focus_positions["abs_position_usd"] > 0]
        .groupby("user")["liq_distance_pct"]
        .median()
        .rename("median_liq_distance_pct")
    )
    active_minutes = active_wallet_minute.groupby("user").size().rename("active_minutes")
    trade_minutes = (
        wallet_trade_flags.loc[wallet_trade_flags["trade_flag"]]
        .groupby("user")
        .size()
        .rename("trade_minutes")
    )
    stability = (1 - safe_divide(trade_minutes, active_minutes)).rename("stability")
    net_exposure_ratio = active_wallet_minute.groupby("user")["net_exposure_ratio"].mean().rename("net_exposure_ratio")
    cross_symbol_offset = active_wallet_minute.groupby("user")["offset_flag"].mean().rename("cross_symbol_offset")
    dominant_symbol = (
        active_wallet_minute.groupby("user")["dominant_symbol"]
        .agg(lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[-1])
        .rename("dominant_symbol")
    )

    symbol_hold_rows = []
    for (user, symbol), g in focus_positions.groupby(["user", "symbol"], sort=False):
        g = g.sort_values("event_time")
        trade_times = g.loc[g["is_trade_minute"], "event_time"].drop_duplicates()
        weight = float(g["abs_position_usd"].mean())
        if weight <= 0:
            continue
        if len(trade_times) >= 2:
            hold = float(trade_times.diff().dropna().dt.total_seconds().div(3600).median())
        else:
            hold = float(max(g["minute"].nunique(), 1) / 60.0)
        symbol_hold_rows.append(
            {
                "user": user,
                "symbol": symbol,
                "holding_time_hours": hold,
                "weight": weight,
            }
        )

    hold_df = pd.DataFrame(symbol_hold_rows)
    if len(hold_df):
        holding_time_hours = (
            hold_df.groupby("user")
            .apply(lambda g: weighted_average(g["holding_time_hours"], g["weight"]))
            .rename("holding_time_hours")
        )
    else:
        holding_time_hours = pd.Series(dtype="float64", name="holding_time_hours")

    wallet_features = pd.concat(
        [
            avg_abs_exposure,
            turnover,
            holding_time_hours,
            avg_leverage,
            pnl_volatility,
            median_liq_distance_pct,
            stability,
            net_exposure_ratio,
            cross_symbol_offset,
            active_minutes,
            trade_minutes,
            dominant_symbol,
        ],
        axis=1,
    )
    wallet_features["trade_minutes"] = wallet_features["trade_minutes"].fillna(0)
    wallet_features["turnover_usd"] = wallet_features["turnover_usd"].fillna(0.0)
    wallet_features["stability"] = wallet_features["stability"].clip(lower=0, upper=1)
    wallet_features["abs_net_exposure_ratio"] = wallet_features["net_exposure_ratio"].abs()
    wallet_features = wallet_features.replace([np.inf, -np.inf], np.nan)
    return wallet_features.sort_values("avg_abs_exposure", ascending=False)


def score_wallet_archetypes(wallet_features: pd.DataFrame) -> pd.DataFrame:
    wf = wallet_features.copy()
    rank_inputs = [
        "avg_abs_exposure",
        "turnover_usd",
        "holding_time_hours",
        "avg_leverage",
        "pnl_volatility",
        "median_liq_distance_pct",
        "stability",
        "cross_symbol_offset",
        "abs_net_exposure_ratio",
    ]
    for col in rank_inputs:
        series = wf[col].fillna(wf[col].median())
        wf[f"{col}_rank"] = series.rank(pct=True)

    wf["high_conviction_score"] = (
        wf["avg_abs_exposure_rank"] + wf["holding_time_hours_rank"] + wf["stability_rank"] - wf["turnover_usd_rank"]
    )
    wf["fast_speculator_score"] = (
        wf["turnover_usd_rank"] + wf["avg_leverage_rank"] + wf["abs_net_exposure_ratio_rank"] - wf["holding_time_hours_rank"]
    )
    wf["hedger_like_score"] = (
        wf["cross_symbol_offset_rank"] + wf["stability_rank"] - wf["abs_net_exposure_ratio_rank"]
    )
    wf["fragile_score"] = (
        (1 - wf["median_liq_distance_pct_rank"]) + wf["avg_leverage_rank"] + wf["pnl_volatility_rank"]
    )
    wf["pnl_swing_score"] = wf["pnl_volatility_rank"] + wf["turnover_usd_rank"]

    archetype_score_cols = [
        "high_conviction_score",
        "fast_speculator_score",
        "hedger_like_score",
        "fragile_score",
        "pnl_swing_score",
    ]
    wf["dominant_archetype"] = wf[archetype_score_cols].idxmax(axis=1).str.replace("_score", "", regex=False)
    sorted_scores = np.sort(wf[archetype_score_cols].to_numpy(dtype=float), axis=1)
    wf["archetype_margin"] = sorted_scores[:, -1] - sorted_scores[:, -2]
    return wf


def choose_selected_wallets(wallet_archetypes: pd.DataFrame, selected_archetypes) -> tuple[set[str], str]:
    tokens = [str(x).strip().lower() for x in selected_archetypes if str(x).strip()]
    if not tokens or "all" in tokens:
        return set(wallet_archetypes["user"].astype(str)), "all"

    invalid = sorted(set(tokens) - set(ARCHETYPE_ORDER))
    if invalid:
        raise ValueError(f"Unknown archetypes in SELECTED_ARCHETYPES: {invalid}")

    selected_wallets = set(
        wallet_archetypes.loc[
            wallet_archetypes["dominant_archetype"].astype(str).isin(tokens),
            "user",
        ].astype(str)
    )
    return selected_wallets, "|".join(tokens)


def aggregate_alerts_1m(alerts_df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "minute_utc",
        "time_ms",
        "whale_alert_count",
        "whale_alert_user_n",
        "whale_alert_notional_abs",
        "whale_alert_notional_signed",
        "whale_alert_pos_size_abs",
        "whale_alert_pos_size_signed",
        "whale_alert_open_count",
        "whale_alert_close_count",
        "whale_alert_action1",
        "whale_alert_action1_count",
        "whale_alert_action2",
        "whale_alert_action2_count",
    ]
    if alerts_df.empty:
        return pd.DataFrame(columns=columns)

    records = []
    for minute_utc, g in alerts_df.groupby("minute_utc", sort=True):
        action_counts = g["position_action_label"].fillna("unknown").astype(str).value_counts()
        record = {
            "minute_utc": minute_utc,
            "whale_alert_count": int(len(g)),
            "whale_alert_user_n": int(g["user"].nunique()),
            "whale_alert_notional_abs": float(g["abs_notional_usd"].fillna(0.0).sum()),
            "whale_alert_notional_signed": float(g["signed_notional_usd"].fillna(0.0).sum()),
            "whale_alert_pos_size_abs": float(g["position_size"].abs().fillna(0.0).sum()),
            "whale_alert_pos_size_signed": float(g["position_size"].fillna(0.0).sum()),
            "whale_alert_open_count": int((g["position_action_label"] == "open").sum()),
            "whale_alert_close_count": int((g["position_action_label"] == "close").sum()),
            "whale_alert_action1": action_counts.index[0] if len(action_counts) >= 1 else pd.NA,
            "whale_alert_action1_count": int(action_counts.iloc[0]) if len(action_counts) >= 1 else 0,
            "whale_alert_action2": action_counts.index[1] if len(action_counts) >= 2 else pd.NA,
            "whale_alert_action2_count": int(action_counts.iloc[1]) if len(action_counts) >= 2 else 0,
        }
        records.append(record)

    out = pd.DataFrame.from_records(records).sort_values("minute_utc").reset_index(drop=True)
    out["time_ms"] = datetime_to_time_ms(out["minute_utc"])
    return out[columns]


def aggregate_positions_1m(positions_df: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "minute_utc",
        "time_ms",
        "whale_pos_rows",
        "whale_pos_wallet_n",
        "whale_pos_active_wallet_n",
        "whale_pos_size_abs",
        "whale_pos_size_signed",
        "whale_pos_notional_abs",
        "whale_pos_notional_signed",
        "whale_pos_pnl_sum",
        "whale_pos_lev_wavg",
    ]
    if positions_df.empty:
        return pd.DataFrame(columns=columns)

    minute_state = (
        positions_df.sort_values(["event_time", "user", "symbol"])
        .groupby(["minute_utc", "user", "symbol"], as_index=False)
        .tail(1)
        .copy()
    )

    records = []
    for minute_utc, g in minute_state.groupby("minute_utc", sort=True):
        abs_pos = g["abs_position_usd"].fillna(0.0)
        weights = abs_pos[abs_pos > 0]
        lev_values = g.loc[weights.index, "leverage"] if len(weights) else pd.Series(dtype="float64")
        lev_wavg = weighted_average(lev_values, weights) if len(weights) else np.nan
        records.append(
            {
                "minute_utc": minute_utc,
                "whale_pos_rows": int(len(g)),
                "whale_pos_wallet_n": int(g["user"].nunique()),
                "whale_pos_active_wallet_n": int((abs_pos > 0).sum()),
                "whale_pos_size_abs": float(g["position_size"].abs().fillna(0.0).sum()),
                "whale_pos_size_signed": float(g["position_size"].fillna(0.0).sum()),
                "whale_pos_notional_abs": float(abs_pos.sum()),
                "whale_pos_notional_signed": float(g["signed_position_usd"].fillna(0.0).sum()),
                "whale_pos_pnl_sum": float(g["unrealized_pnl"].fillna(0.0).sum()),
                "whale_pos_lev_wavg": lev_wavg,
            }
        )

    out = pd.DataFrame.from_records(records).sort_values("minute_utc").reset_index(drop=True)
    out["time_ms"] = datetime_to_time_ms(out["minute_utc"])
    return out[columns]


def load_market_1m(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    if "time_ms" not in df.columns:
        first_col = df.columns[0]
        first_numeric = pd.to_numeric(df[first_col], errors="coerce")
        if first_numeric.notna().any():
            df = df.rename(columns={first_col: "time_ms"})

    if "time_ms" in df.columns:
        df["time_ms"] = pd.to_numeric(df["time_ms"], errors="coerce")
        df = df.dropna(subset=["time_ms"]).copy()
        df["time_ms"] = df["time_ms"].astype("int64")
        if "time_utc" in df.columns:
            df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce")
        else:
            df["time_utc"] = pd.to_datetime(df["time_ms"], unit="ms", utc=True, errors="coerce")
    elif "time_utc" in df.columns:
        df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce")
        df["time_ms"] = datetime_to_time_ms(df["time_utc"]).astype("Int64")
    else:
        raise ValueError(f"Could not find a time key in {path}")

    df["minute_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce").dt.floor("min")
    df = df.dropna(subset=["minute_utc"]).sort_values("minute_utc")
    df = df.drop_duplicates(subset=["minute_utc"], keep="last").reset_index(drop=True)
    return df


## 2) Load Raw Whale Data And Run First-Pass Checks

This first pass does four things:

1. Loads and standardizes the raw alerts parquet
2. Scans the large positions parquet in batches
3. Reports rough data quality checks and symbol coverage
4. Builds the wallet universe used for archetype scoring


In [ ]:
alerts_raw = pd.read_parquet(ALERTS_PATH)
alerts = standardize_alerts(alerts_raw)
alerts["minute_utc"] = alerts["event_time"].dt.floor("min")
alerts["time_ms"] = datetime_to_time_ms(alerts["minute_utc"])

assert str(alerts["event_time"].dtype).endswith("UTC]"), "alerts event_time must be UTC"

alert_duplicate_rows = int(alerts.duplicated(subset=["user", "symbol", "event_time"]).sum())
alert_missing_critical = {
    "event_time_missing": int(alerts["event_time"].isna().sum()),
    "user_missing": int(alerts["user"].isna().sum()),
    "symbol_missing": int(alerts["symbol"].isna().sum()),
}

position_read_columns = [
    "ingested_at",
    "user",
    "symbol",
    "position_size",
    "entry_price",
    "mark_price",
    "liq_price",
    "leverage",
    "margin_balance",
    "position_value_usd",
    "unrealized_pnl",
    "funding_fee",
    "margin_mode",
    "create_time",
    "update_time",
]

position_rows_after_standardization = 0
position_raw_duplicate_estimate = 0
positions_min_ts = None
positions_max_ts = None
position_symbol_counts = defaultdict(int)

focus_exposure_sum = defaultdict(float)
focus_exposure_count = defaultdict(int)
focus_snapshot_count = defaultdict(int)
focus_turnover_sum = defaultdict(float)
focus_last_signed = {}

for raw_chunk in iter_parquet_batches(POSITIONS_PATH, columns=position_read_columns, batch_size=POSITION_BATCH_SIZE):
    raw_dedup_keys = [c for c in ["user", "symbol", "update_time"] if c in raw_chunk.columns]
    if len(raw_dedup_keys) == 3:
        position_raw_duplicate_estimate += int(raw_chunk.duplicated(subset=raw_dedup_keys).sum())

    chunk = standardize_positions(raw_chunk)
    if chunk.empty:
        continue

    position_rows_after_standardization += len(chunk)
    chunk_min = chunk["event_time"].min()
    chunk_max = chunk["event_time"].max()
    positions_min_ts = chunk_min if positions_min_ts is None else min(positions_min_ts, chunk_min)
    positions_max_ts = chunk_max if positions_max_ts is None else max(positions_max_ts, chunk_max)

    for key, value in chunk["symbol"].value_counts().items():
        position_symbol_counts[str(key)] += int(value)

    focus_chunk = chunk.loc[
        chunk["symbol"].isin(FOCUS_SYMBOLS_FOR_ARCHETYPES),
        ["event_time", "user", "symbol", "abs_position_usd", "signed_position_usd"],
    ].copy()
    if focus_chunk.empty:
        continue

    for user, value in focus_chunk.groupby("user")["abs_position_usd"].sum().items():
        focus_exposure_sum[str(user)] += float(value)
    for user, value in focus_chunk.groupby("user")["abs_position_usd"].count().items():
        focus_exposure_count[str(user)] += int(value)
    for user, value in focus_chunk.groupby("user").size().items():
        focus_snapshot_count[str(user)] += int(value)

    turnover_chunk = update_turnover_from_chunk(focus_chunk, focus_last_signed)
    for user, value in turnover_chunk.items():
        focus_turnover_sum[str(user)] += float(value)

focus_avg_abs = (pd.Series(focus_exposure_sum) / pd.Series(focus_exposure_count)).sort_values(ascending=False)
focus_turnover = pd.Series(focus_turnover_sum).sort_values(ascending=False)
focus_snapshot_count = pd.Series(focus_snapshot_count).sort_values(ascending=False)

wallet_pool = sorted(
    set(focus_avg_abs.head(TOP_EXPOSURE_WALLETS).index) | set(focus_turnover.head(TOP_TURNOVER_WALLETS).index)
)
wallet_pool = [wallet for wallet in wallet_pool if int(focus_snapshot_count.get(wallet, 0)) >= MIN_WALLET_SNAPSHOTS]

if not wallet_pool and len(focus_snapshot_count):
    wallet_pool = focus_snapshot_count[focus_snapshot_count >= MIN_WALLET_SNAPSHOTS].head(250).index.tolist()

overview_stats = pd.DataFrame(
    [
        {
            "dataset": "whale_alerts",
            "rows": len(alerts),
            "start_utc": alerts["event_time"].min(),
            "end_utc": alerts["event_time"].max(),
            "duplicate_rows": alert_duplicate_rows,
        },
        {
            "dataset": "whale_positions",
            "rows": int(position_rows_after_standardization),
            "start_utc": positions_min_ts,
            "end_utc": positions_max_ts,
            "duplicate_rows": int(position_raw_duplicate_estimate),
        },
    ]
)

alert_symbol_counts = alerts["symbol"].value_counts().sort_values(ascending=False)
position_symbol_counts = pd.Series(position_symbol_counts).sort_values(ascending=False)

display(overview_stats)
display(pd.DataFrame([alert_missing_critical]))
display(alert_symbol_counts.head(12).rename("alert_rows").to_frame())
display(position_symbol_counts.head(12).rename("position_rows").to_frame())
print(
    {
        "wallet_pool_size": len(wallet_pool),
        "top_focus_exposure_wallet": focus_avg_abs.index[0] if len(focus_avg_abs) else None,
        "top_focus_turnover_wallet": focus_turnover.index[0] if len(focus_turnover) else None,
    }
)


## 3) Build Wallet Archetypes

This section reuses the same 5-archetype heuristic from `Hyperliquid_Whale_Wallet_Exploration.ipynb`.

Important detail:
- archetypes are computed from **multi-symbol wallet behavior** on the focus universe
- the final handoff table is still **BTC-only** at the 1-minute level


In [ ]:
if not wallet_pool:
    raise ValueError("wallet_pool is empty; widen the wallet universe thresholds or inspect the raw data.")

focus_positions_parts = []
for raw_chunk in iter_parquet_batches(POSITIONS_PATH, columns=position_read_columns, batch_size=POSITION_BATCH_SIZE):
    chunk = standardize_positions(raw_chunk)
    if chunk.empty:
        continue
    chunk = chunk.loc[
        chunk["symbol"].isin(FOCUS_SYMBOLS_FOR_ARCHETYPES) & chunk["user"].isin(wallet_pool),
        [
            "event_time",
            "user",
            "symbol",
            "mark_price",
            "leverage",
            "unrealized_pnl",
            "abs_position_usd",
            "signed_position_usd",
            "liq_distance_pct",
            "position_size",
        ],
    ].copy()
    if chunk.empty:
        continue
    chunk["minute"] = chunk["event_time"].dt.floor("min")
    focus_positions_parts.append(chunk)

if not focus_positions_parts:
    raise ValueError("No focus position rows were collected for the wallet archetype stage.")

focus_positions = pd.concat(focus_positions_parts, ignore_index=True)
focus_positions = focus_positions.sort_values(["user", "symbol", "event_time"]).reset_index(drop=True)
focus_positions["delta_position_usd"] = focus_positions.groupby(["user", "symbol"])["signed_position_usd"].diff().fillna(0.0)
focus_positions["abs_delta_position_usd"] = focus_positions["delta_position_usd"].abs()
focus_positions["is_trade_minute"] = focus_positions["abs_delta_position_usd"] >= TRADE_EPS_USD
focus_positions["active_flag"] = focus_positions["abs_position_usd"] > 0

minute_symbol_state = (
    focus_positions.sort_values(["event_time", "user", "symbol"])
    .groupby(["minute", "user", "symbol"], as_index=False)
    .tail(1)
    .copy()
)
minute_symbol_state["weighted_leverage_num"] = (
    minute_symbol_state["leverage"].fillna(0.0) * minute_symbol_state["abs_position_usd"].fillna(0.0)
)

wallet_minute_core = minute_symbol_state.assign(
    has_long=minute_symbol_state["signed_position_usd"] > 0,
    has_short=minute_symbol_state["signed_position_usd"] < 0,
).groupby(["minute", "user"], as_index=False).agg(
    total_signed_exposure=("signed_position_usd", "sum"),
    total_abs_exposure=("abs_position_usd", "sum"),
    total_unrealized_pnl=("unrealized_pnl", "sum"),
    weighted_leverage_num=("weighted_leverage_num", "sum"),
    has_long=("has_long", "any"),
    has_short=("has_short", "any"),
)

dominant_symbol_rows = (
    minute_symbol_state.sort_values(["minute", "user", "abs_position_usd", "event_time"])
    .groupby(["minute", "user"], as_index=False)
    .tail(1)
    [["minute", "user", "symbol", "mark_price"]]
    .rename(columns={"symbol": "dominant_symbol", "mark_price": "dominant_mark_price"})
)

wallet_trade_minute = (
    focus_positions.groupby(["minute", "user"], as_index=False)["is_trade_minute"]
    .max()
    .rename(columns={"is_trade_minute": "trade_flag"})
)

wallet_minute = wallet_minute_core.merge(dominant_symbol_rows, on=["minute", "user"], how="left")
wallet_minute = wallet_minute.merge(wallet_trade_minute, on=["minute", "user"], how="left")
wallet_minute["avg_leverage"] = safe_divide(wallet_minute["weighted_leverage_num"], wallet_minute["total_abs_exposure"])
wallet_minute["offset_flag"] = wallet_minute["has_long"] & wallet_minute["has_short"]
wallet_minute["net_exposure_ratio"] = safe_divide(
    wallet_minute["total_signed_exposure"], wallet_minute["total_abs_exposure"]
).clip(-1, 1)
wallet_minute["trade_flag"] = wallet_minute["trade_flag"].fillna(False)

wallet_features = compute_wallet_features(focus_positions, wallet_minute)
wallet_features = score_wallet_archetypes(wallet_features)
wallet_archetypes = wallet_features.reset_index()
if "index" in wallet_archetypes.columns:
    wallet_archetypes = wallet_archetypes.rename(columns={"index": "user"})

wallet_archetypes["dominant_archetype"] = wallet_archetypes["dominant_archetype"].astype(str)
assert set(wallet_archetypes["dominant_archetype"].dropna().unique()).issubset(set(ARCHETYPE_ORDER))

selected_wallets, archetype_filter_label = choose_selected_wallets(wallet_archetypes, SELECTED_ARCHETYPES)
wallet_archetypes["selected_for_output"] = wallet_archetypes["user"].astype(str).isin(selected_wallets)

if [str(x).lower() for x in SELECTED_ARCHETYPES] == ["all"]:
    assert len(selected_wallets) == wallet_archetypes["user"].nunique()
else:
    selected_archetypes_observed = set(
        wallet_archetypes.loc[
            wallet_archetypes["user"].astype(str).isin(selected_wallets),
            "dominant_archetype",
        ]
    )
    assert selected_archetypes_observed.issubset(set(a.lower() for a in SELECTED_ARCHETYPES))

display(
    pd.DataFrame(
        {
            "metric": ["wallet_pool_size", "scored_wallets", "selected_wallets", "archetype_filter"],
            "value": [len(wallet_pool), wallet_archetypes["user"].nunique(), len(selected_wallets), archetype_filter_label],
        }
    )
)
display(
    wallet_archetypes["dominant_archetype"]
    .value_counts()
    .reindex(ARCHETYPE_ORDER)
    .fillna(0)
    .rename("wallets")
    .to_frame()
)
display(wallet_archetypes.head(10))


## 4) Aggregate Selected Whale Data To 1-Minute BTC Features

This is the main handoff table.

Workflow:
1. keep only the selected wallet archetypes
2. keep only `TARGET_SYMBOL`
3. aggregate alerts and positions separately to `1min`
4. outer-join them into one rough whale base table


In [ ]:
wallet_lookup = wallet_archetypes[["user", "dominant_archetype"]].copy()

alerts_target = alerts.loc[
    alerts["symbol"].eq(TARGET_SYMBOL) & alerts["user"].astype(str).isin(selected_wallets)
].copy()
alerts_target = alerts_target.merge(wallet_lookup, on="user", how="left")
alerts_target["minute_utc"] = alerts_target["event_time"].dt.floor("min")

target_position_parts = []
for raw_chunk in iter_parquet_batches(POSITIONS_PATH, columns=position_read_columns, batch_size=POSITION_BATCH_SIZE):
    chunk = standardize_positions(raw_chunk)
    if chunk.empty:
        continue
    chunk = chunk.loc[
        chunk["symbol"].eq(TARGET_SYMBOL) & chunk["user"].astype(str).isin(selected_wallets),
        [
            "event_time",
            "user",
            "symbol",
            "position_size",
            "abs_position_usd",
            "signed_position_usd",
            "unrealized_pnl",
            "leverage",
        ],
    ].copy()
    if chunk.empty:
        continue
    chunk["minute_utc"] = chunk["event_time"].dt.floor("min")
    target_position_parts.append(chunk)

positions_target = pd.concat(target_position_parts, ignore_index=True) if target_position_parts else pd.DataFrame(
    columns=["event_time", "user", "symbol", "position_size", "abs_position_usd", "signed_position_usd", "unrealized_pnl", "leverage", "minute_utc"]
)
if not positions_target.empty:
    positions_target = positions_target.merge(wallet_lookup, on="user", how="left")

alert_1m = aggregate_alerts_1m(alerts_target)
position_1m = aggregate_positions_1m(positions_target)

whale_features_1m = alert_1m.merge(position_1m, on=["minute_utc", "time_ms"], how="outer")
whale_features_1m = whale_features_1m.sort_values("minute_utc").reset_index(drop=True)
whale_features_1m["target_symbol"] = TARGET_SYMBOL
whale_features_1m["archetype_filter_applied"] = archetype_filter_label
whale_features_1m["selected_wallet_count"] = len(selected_wallets)

assert whale_features_1m["minute_utc"].is_unique, "whale_features_1m must have unique minute_utc keys"

display(
    pd.DataFrame(
        {
            "metric": [
                "alerts_target_rows",
                "positions_target_rows",
                "whale_feature_rows",
                "whale_feature_start_utc",
                "whale_feature_end_utc",
            ],
            "value": [
                len(alerts_target),
                len(positions_target),
                len(whale_features_1m),
                whale_features_1m["minute_utc"].min() if len(whale_features_1m) else pd.NaT,
                whale_features_1m["minute_utc"].max() if len(whale_features_1m) else pd.NaT,
            ],
        }
    )
)
display(whale_features_1m.head(10))


## 5) Merge To Market `cleaned_1m.csv`

This final join is only for handoff convenience.

- the whale base table is already usable on its own
- if `data/cleaned/cleaned_1m.csv` is available, we left-join so market row count is preserved


In [ ]:
market_1m = None
market_with_whales = None
coverage_summary = None
whale_null_counts = None

if MARKET_1M_PATH.exists():
    market_1m = load_market_1m(MARKET_1M_PATH)
    whale_merge_table = whale_features_1m.drop(columns=["time_ms"], errors="ignore")
    market_with_whales = market_1m.merge(whale_merge_table, on="minute_utc", how="left")
    market_with_whales["archetype_filter_applied"] = archetype_filter_label

    assert len(market_with_whales) == len(market_1m), "left merge must preserve cleaned_1m row count"

    whale_value_cols = [
        c
        for c in whale_merge_table.columns
        if c not in {"minute_utc", "archetype_filter_applied", "target_symbol", "selected_wallet_count"}
    ]
    pct_covered = (
        round(100 * market_with_whales[whale_value_cols].notna().any(axis=1).mean(), 2)
        if whale_value_cols
        else np.nan
    )
    coverage_summary = pd.DataFrame(
        [
            {
                "market_rows": len(market_1m),
                "whale_rows": len(whale_features_1m),
                "merged_rows": len(market_with_whales),
                "market_start_utc": market_1m["minute_utc"].min(),
                "market_end_utc": market_1m["minute_utc"].max(),
                "whale_start_utc": whale_features_1m["minute_utc"].min() if len(whale_features_1m) else pd.NaT,
                "whale_end_utc": whale_features_1m["minute_utc"].max() if len(whale_features_1m) else pd.NaT,
                "pct_market_minutes_with_any_whale_feature": pct_covered,
            }
        ]
    )
    whale_null_counts = market_with_whales[whale_value_cols].isna().sum().sort_values(ascending=False).rename("null_count").to_frame()

    display(coverage_summary)
    display(whale_null_counts.head(20))
    display(market_with_whales.head(10))
else:
    print(f"Skipping market merge because the file is missing: {MARKET_1M_PATH}")


## 6) Save Outputs

The notebook writes the archetype lookup, the rough 1-minute whale base table, and the merged market table when available.


In [ ]:
wallet_archetypes_path = OUTPUT_DIR / "wallet_archetypes.csv"
whale_features_path = OUTPUT_DIR / "whale_features_1m.csv"
merged_market_path = MERGED_OUTPUT_DIR / "cleaned_1m_with_whales.csv"

wallet_archetypes.to_csv(wallet_archetypes_path, index=False)
whale_features_1m.to_csv(whale_features_path, index=False)

merged_status = "skipped"
if market_with_whales is not None:
    market_with_whales.to_csv(merged_market_path, index=False)
    merged_status = "written"

saved_outputs = pd.DataFrame(
    [
        {"output": "wallet_archetypes", "path": str(wallet_archetypes_path), "status": "written"},
        {"output": "whale_features_1m", "path": str(whale_features_path), "status": "written"},
        {"output": "cleaned_1m_with_whales", "path": str(merged_market_path), "status": merged_status},
    ]
)
display(saved_outputs)
